# Task 3 - Retrieval: Popularity, Random, MF-BPR, and Two-Tower

This notebook is the executable Task 3 pipeline.

It keeps concerns separated:
- model code lives in `scripts/models.py` and `scripts/mf_bpr.py`
- evaluation logic lives in `scripts/evaluate_task3.py`
- plot generation lives in `scripts/plot_task3_results.py`
- this notebook orchestrates the workflow and records the results

Run this notebook on a GPU runtime if you want the fastest MF-BPR and two-tower training. The code still works on CPU with a smaller `max_train_positives` cap, but two-tower is intended for Colab/GPU.

## Colab setup

Use these cells first when running in Google Colab. They clone the repo at the task branch and add the repo root to `sys.path`.

In [1]:
# In Colab, replace REPO_URL with your remote repository URL.
REPO_URL = "https://github.com/truongtan29602/steam-recsys-pipeline.git"
BRANCH = "task3-MF-BPR-Two-Tower"
CLONE_DIR = "/content/steam-recsys-pipeline"

!rm -rf $CLONE_DIR
!git clone --branch $BRANCH $REPO_URL $CLONE_DIR

Cloning into '/content/steam-recsys-pipeline'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 138 (delta 1), reused 3 (delta 1), pack-reused 130 (from 3)
Receiving objects: 100% (138/138), 125.68 MiB | 15.42 MiB/s, done.
Resolving deltas: 100% (25/25), done.
Updating files: 100% (54/54), done.


In [2]:
import sys
from pathlib import Path

REPO_ROOT = Path('/content/steam-recsys-pipeline')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(REPO_ROOT)
print(sys.path[0])

/content/steam-recsys-pipeline
/content/steam-recsys-pipeline


In [3]:
from pathlib import Path
import json
import pandas as pd

from scripts.evaluate_task3 import build_ground_truth, build_history, evaluate
from scripts.mf_bpr import MFBPRConfig
from scripts.two_tower import TwoTowerConfig
from scripts.plot_task3_results import load_results, plot_bars, plot_latency, plot_tradeoff

DATA_DIR = REPO_ROOT
OUTPUT_DIR = Path('outputs/task3')
PLOTS_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
print('data dir', DATA_DIR)
print('output dir', OUTPUT_DIR)
print('plots dir', PLOTS_DIR)

data dir /content/steam-recsys-pipeline
output dir outputs/task3
plots dir outputs/task3/plots


## 1. Load and inspect the processed splits

The project uses time-based splits already prepared in parquet format.

In [5]:
train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
val_df = pd.read_parquet(DATA_DIR / 'validation.parquet')
test_df = pd.read_parquet(DATA_DIR / 'test.parquet')

display(train_df.head())
print('train', train_df.shape)
print('validation', val_df.shape)
print('test', test_df.shape)
print('train positives', int(train_df['is_positive'].sum()))
print('validation positives', int(val_df['is_positive'].sum()))
print('test positives', int(test_df['is_positive'].sum()))

,user_id,item_id,hours,event_time,is_positive,text_length,early_access_review,products_owned,found_funny,received_for_free
0,Manatee Malteser,239140,25.6,2015-04-12,True,58,False,83.0,0,False
1,BlackDoge,298110,242.3,2015-04-12,True,21,False,36.0,0,False
2,GreyWalker Zero,246090,5.1,2015-04-12,True,1617,False,347.0,0,False
3,Tiff,353560,0.8,2015-04-12,False,303,False,54.0,1,False
4,shery360,261570,7.7,2015-04-12,True,37,False,4.0,0,False


train (3600000, 10)
validation (400000, 10)
test (1000000, 10)
train positives 3198885
validation positives 354018
test positives 904060


## 2. Build histories and ground truth

The evaluation uses training history for validation and training + validation history for test, matching the project protocol.

In [6]:
val_history = build_history(train_df)
test_history = build_history(train_df, val_df)
val_gt = build_ground_truth(val_df)
test_gt = build_ground_truth(test_df)

val_users = [u for u, pos in val_gt.items() if pos]
test_users = [u for u, pos in test_gt.items() if pos]
catalog = set(train_df['item_id'].astype(str).unique())

print('catalog size', len(catalog))
print('validation users', len(val_users))
print('test users', len(test_users))

catalog size 12146
validation users 251589
test users 592796


## 3. Configure MF-BPR and Two-Tower

MF-BPR is implemented from scratch in NumPy. Two-Tower is implemented in the local module as a PyTorch model. These configs are the main place to tune dimensions, learning rate, regularization, epochs, and the training cap for local GPU/CPU experiments.

In [7]:
config = MFBPRConfig(
    n_factors=64,
    learning_rate=0.05,
    reg=0.0025,
    epochs=5,
    batch_size=2048,
    seed=42,
    device='cuda',
)
max_train_positives = 200000
print(config)
print('max_train_positives', max_train_positives)

MFBPRConfig(n_factors=64, learning_rate=0.05, reg=0.0025, epochs=5, num_negatives=1, batch_size=2048, seed=42, device='cuda')
max_train_positives 200000


In [8]:
two_tower_config = TwoTowerConfig(
    user_dim=64,
    item_dim=64,
    hidden_dim=128,
    batch_size=1024,
    epochs=5,
    learning_rate=1e-3,
    temperature=0.07,
    seed=42,
    device='cuda',
)
print(two_tower_config)

TwoTowerConfig(user_dim=64, item_dim=64, hidden_dim=128, batch_size=1024, epochs=5, learning_rate=0.001, temperature=0.07, seed=42, device='cuda')


## 4. Fit and evaluate the full Task 3 comparison

This step trains MF-BPR and two-tower, then evaluates all four retrievers on validation and test. The returned structure is also saved to JSON for plot generation and analysis.

In [9]:
results = evaluate(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    k=20,
    ndcg_k=10,
    mf_config=config,
    two_tower_config=two_tower_config,
    max_train_positives=max_train_positives,
)

results

{'validation': [{'model': 'Random',
   'users': 251589,
   'recall@20': 0.001347326971410298,
   'ndcg@10': 0.00031944804554624257,
   'coverage@20': 1.0,
   'recommendation_time_sec': 3.6098068629999034},
  {'model': 'Popularity',
   'users': 251589,
   'recall@20': 0.17522717803824261,
   'ndcg@10': 0.05241603664953573,
   'coverage@20': 0.004034249958834184,
   'recommendation_time_sec': 3.259317536000026},
  {'model': 'MF-BPR',
   'users': 251589,
   'recall@20': 2.0657732640370994e-05,
   'ndcg@10': 8.446627200612581e-06,
   'coverage@20': 0.5078215050222296,
   'recommendation_time_sec': 4.007363099000031},
  {'model': 'Two-Tower',
   'users': 251589,
   'recall@20': 2.5968278952312434e-05,
   'ndcg@10': 2.4163053426717413e-06,
   'coverage@20': 0.6150996212744937,
   'recommendation_time_sec': 19.665894440000102}],
 'test': [{'model': 'Random',
   'users': 592796,
   'recall@20': 0.0012024586918164143,
   'ndcg@10': 0.0002960893653446251,
   'coverage@20': 1.0,
   'recommendatio

## 5. Turn results into a comparison table

This is the table you can later paste into `ANALYSIS.md` or the README.

In [10]:
results_df = load_results(OUTPUT_DIR / 'task3_results.json')
display(results_df)

,model,users,recall@20,ndcg@10,coverage@20,recommendation_time_sec,split
0,Random,251589,0.001347,0.000319,1.000000,3.609807,validation
1,Popularity,251589,0.175227,0.052416,0.004034,3.259318,validation
2,MF-BPR,251589,0.000021,0.000008,0.507822,4.007363,validation
3,Two-Tower,251589,0.000026,0.000002,0.615100,19.665894,validation
4,Random,592796,0.001202,0.000296,1.000000,8.601838,test
5,Popularity,592796,0.169934,0.054589,0.004528,10.367353,test
6,MF-BPR,592796,0.000019,0.000006,0.514902,6.082358,test
7,Two-Tower,592796,0.000024,0.000003,0.628355,33.956804,test


## 6. Generate plots

The notebook generates the same plots as the standalone script so the full pipeline remains reproducible from the notebook alone.

In [11]:
plot_bars(results_df, 'recall@20', PLOTS_DIR / 'recall_at_20.png')
plot_bars(results_df, 'ndcg@10', PLOTS_DIR / 'ndcg_at_10.png')
plot_bars(results_df, 'coverage@20', PLOTS_DIR / 'coverage_at_20.png')
plot_latency(results_df, PLOTS_DIR / 'latency.png')
plot_tradeoff(results_df, PLOTS_DIR / 'coverage_vs_recall.png')

print('plots saved to', PLOTS_DIR)

plots saved to outputs/task3/plots


## 7. Save a compact summary

This creates a notebook-friendly artifact you can cite in the report.

In [12]:
summary = {
    'config': config.__dict__,
    'max_train_positives': max_train_positives,
    'results': results,
}
with open(OUTPUT_DIR / 'task3_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=str)

print('saved', OUTPUT_DIR / 'task3_summary.json')

saved outputs/task3/task3_summary.json
